# 🎬 나래 채널 자동 편집 패턴 분석

**사용법:** 위에서 아래로 셀을 순서대로 실행하세요 (셀 클릭 후 `Shift+Enter`)

1. 패키지 설치
2. 채널 URL 입력
3. 분석 실행
4. 결과 확인

In [ ]:
# ✅ 셀 1: 패키지 설치 (처음 한 번만, 2~3분 소요)
!pip install -q yt-dlp librosa soundfile numpy scipy

In [ ]:
# ✅ 셀 2: 채널 설정

CHANNEL_URL = "https://youtube.com/@leenaraek"  # ← 채널 URL
ANALYZE_COUNT = 20  # 분석할 영상 수 (많을수록 정확, 시간도 오래 걸림)
DOWNLOAD_DIR = "/content/videos"

import os
os.makedirs(DOWNLOAD_DIR, exist_ok=True)
print(f"채널: {CHANNEL_URL}")
print(f"분석 영상 수: {ANALYZE_COUNT}개")

In [ ]:
# ✅ 셀 3: 채널 영상 목록 가져오기
import subprocess, json

print("채널 영상 목록 수집 중...")
result = subprocess.run([
    "yt-dlp", "--flat-playlist", "--dump-single-json",
    "--no-warnings", CHANNEL_URL
], capture_output=True, text=True, timeout=120)

data = json.loads(result.stdout)
all_videos = [
    {
        "id": e.get("id"),
        "title": e.get("title"),
        "duration": e.get("duration") or 0,
        "url": f"https://youtube.com/watch?v={e.get('id')}"
    }
    for e in data.get("entries", []) if e and e.get("id")
]

print(f"\n총 {len(all_videos)}개 영상 발견")
print("\n최근 영상 5개:")
for v in all_videos[:5]:
    m, s = divmod(v['duration'], 60)
    print(f"  [{m}:{s:02d}] {v['title'][:50]}")

In [ ]:
# ✅ 셀 4: 분석할 영상 선택
# 짧은/중간/긴 영상 골고루 선택해서 패턴 대표성 높이기

valid = [v for v in all_videos if 60 < v["duration"] < 3600]  # 1분~1시간
valid.sort(key=lambda v: v["duration"])

n = min(ANALYZE_COUNT, len(valid))
indices = [int(i * (len(valid) - 1) / (n - 1)) for i in range(n)] if n > 1 else [0]
selected = [valid[i] for i in indices]

print(f"선택된 {len(selected)}개 영상:")
for v in selected:
    m, s = divmod(v['duration'], 60)
    print(f"  [{m}:{s:02d}] {v['title'][:55]}")

In [ ]:
# ✅ 셀 5: 오디오만 다운로드 (영상보다 훨씬 빠름)
from pathlib import Path

def download_audio(video):
    out = Path(DOWNLOAD_DIR) / f"{video['id']}.wav"
    if out.exists():
        print(f"  이미 있음: {video['title'][:40]}")
        return out
    
    result = subprocess.run([
        "yt-dlp",
        "-x", "--audio-format", "wav",
        "--audio-quality", "0",
        "--postprocessor-args", "ffmpeg:-ar 16000 -ac 1",
        "-o", str(Path(DOWNLOAD_DIR) / f"{video['id']}.%(ext)s"),
        "--no-playlist", "--no-warnings",
        video["url"]
    ], capture_output=True, text=True, timeout=300)
    
    if result.returncode == 0 and out.exists():
        print(f"  ✓ {video['title'][:45]}")
        return out
    else:
        print(f"  ✗ 실패: {video['title'][:40]}")
        return None

print(f"오디오 다운로드 시작 ({len(selected)}개)...\n")
downloaded = []
for i, v in enumerate(selected, 1):
    print(f"[{i}/{len(selected)}]", end=" ")
    path = download_audio(v)
    if path:
        downloaded.append({"video": v, "audio_path": str(path)})

print(f"\n완료: {len(downloaded)}/{len(selected)}개")

In [ ]:
# ✅ 셀 6: 각 영상 오디오 분석
import numpy as np
import librosa
from dataclasses import dataclass, asdict

@dataclass
class VideoPattern:
    video_id: str
    title: str
    duration: float
    silence_segments: list       # 무음 구간 목록
    avg_silence_dur: float       # 평균 무음 길이
    silence_ratio: float         # 전체 중 무음 비율
    speech_pace: float           # 분당 말하기 세션 수
    avg_rms_db: float            # 평균 음량
    cut_threshold: float         # 이 영상의 추천 컷 임계값

def analyze_audio(audio_path, video):
    y, sr = librosa.load(audio_path, sr=16000, mono=True)
    total_dur = len(y) / sr

    hop = 512
    rms = librosa.feature.rms(y=y, frame_length=2048, hop_length=hop)[0]
    rms_db = librosa.amplitude_to_db(rms, ref=max(rms.max(), 1e-6))
    times = librosa.frames_to_time(np.arange(len(rms)), sr=sr, hop_length=hop)

    avg_db = float(np.mean(rms_db))
    threshold = max(avg_db - 20, -55)
    is_silent = rms_db < threshold

    # 무음 구간
    silence_segs = []
    in_sil, sil_start = False, 0.0
    for t, sil in zip(times, is_silent):
        if sil and not in_sil:
            in_sil, sil_start = True, t
        elif not sil and in_sil:
            in_sil = False
            dur = t - sil_start
            if dur >= 0.2:
                silence_segs.append({"start": round(sil_start,2), "end": round(t,2), "dur": round(dur,2)})
    if in_sil:
        dur = times[-1] - sil_start
        if dur >= 0.2:
            silence_segs.append({"start": round(sil_start,2), "end": round(times[-1],2), "dur": round(dur,2)})

    # 말하기 세션 수
    is_speech = ~is_silent
    speech_sessions = sum(1 for i in range(1, len(is_speech)) if is_speech[i] and not is_speech[i-1])
    speech_pace = speech_sessions / (total_dur / 60) if total_dur > 0 else 0

    sil_durs = [s["dur"] for s in silence_segs]
    total_sil = sum(sil_durs)

    # 컷 임계값: 무음 길이 상위 40%를 컷 대상으로
    cut_thr = float(np.percentile(sil_durs, 60)) if sil_durs else 0.5

    return VideoPattern(
        video_id=video["id"],
        title=video["title"],
        duration=round(total_dur, 1),
        silence_segments=silence_segs,
        avg_silence_dur=round(float(np.mean(sil_durs)), 3) if sil_durs else 0,
        silence_ratio=round(total_sil / total_dur, 3) if total_dur > 0 else 0,
        speech_pace=round(speech_pace, 1),
        avg_rms_db=round(avg_db, 1),
        cut_threshold=round(cut_thr, 2),
    )

print("오디오 패턴 분석 중...\n")
patterns = []
for item in downloaded:
    v = item["video"]
    print(f"분석: {v['title'][:50]}")
    p = analyze_audio(item["audio_path"], v)
    patterns.append(p)
    print(f"  무음비율={p.silence_ratio*100:.1f}%, 평균무음={p.avg_silence_dur:.2f}초, 컷임계={p.cut_threshold:.2f}초")

print(f"\n분석 완료: {len(patterns)}개")

In [ ]:
# ✅ 셀 7: 채널 전체 편집 패턴 요약

all_sil_durs = []
for p in patterns:
    all_sil_durs.extend([s["dur"] for s in p.silence_segments])

channel_pattern = {
    "analyzed_videos": len(patterns),
    "avg_silence_ratio": round(float(np.mean([p.silence_ratio for p in patterns])), 3),
    "avg_silence_duration": round(float(np.mean(all_sil_durs)), 3) if all_sil_durs else 0,
    "recommended_cut_threshold": round(float(np.percentile(all_sil_durs, 60)), 2) if all_sil_durs else 0.5,
    "avg_speech_pace": round(float(np.mean([p.speech_pace for p in patterns])), 1),
    "avg_rms_db": round(float(np.mean([p.avg_rms_db for p in patterns])), 1),
}

print("="*55)
print("      나래 채널 편집 패턴 분석 결과")
print("="*55)
print(f"  분석한 영상 수      : {channel_pattern['analyzed_videos']}개")
print(f"  평균 무음 비율      : {channel_pattern['avg_silence_ratio']*100:.1f}%")
print(f"  평균 무음 길이      : {channel_pattern['avg_silence_duration']:.2f}초")
print(f"  자동 컷 기준        : {channel_pattern['recommended_cut_threshold']:.2f}초 이상 무음")
print(f"  분당 말하기 세션    : {channel_pattern['avg_speech_pace']:.0f}회")
print(f"  평균 음량           : {channel_pattern['avg_rms_db']:.1f} dB")
print("="*55)
print()
print("영상별 요약:")
print(f"{'제목':40s} {'무음%':6s} {'컷기준':6s}")
print("-"*55)
for p in patterns:
    print(f"{p.title[:40]:40s} {p.silence_ratio*100:5.1f}% {p.cut_threshold:5.2f}s")

In [ ]:
# ✅ 셀 8: 결과 저장 (Google Drive에 저장하고 싶으면 마운트 후 경로 변경)

import json
result_data = {
    "channel_pattern": channel_pattern,
    "individual": [asdict(p) for p in patterns]
}

with open("/content/narae_pattern.json", "w", encoding="utf-8") as f:
    json.dump(result_data, f, ensure_ascii=False, indent=2)

print("저장 완료: /content/narae_pattern.json")
print("\n이 파일을 다운받아두면 다음에 다시 분석 안 해도 됩니다.")

# 파일 다운로드
from google.colab import files
files.download("/content/narae_pattern.json")